# NBA Game Outcome Prediction
## Notebook 1: Random Forest Classifier
**Member 1 Contribution**

This notebook applies a **Random Forest Classifier** to predict whether the home team wins an NBA game based on team performance statistics.

**Dataset:** NBA Games Dataset — https://www.kaggle.com/datasets/nathanlauga/nba-games


## 0. Install Required Packages

In [ ]:
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn']

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('All packages installed successfully!')

## 0.1 Download Dataset

In [ ]:
import os
import subprocess
import sys

if not os.path.exists('games.csv'):
    print('games.csv not found. Downloading dataset...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    path = kagglehub.dataset_download('nathanlauga/nba-games')
    print('Dataset downloaded to:', path)
    # Copy games.csv to current directory
    import shutil
    for root, dirs, files in os.walk(path):
        for file in files:
            if file == 'games.csv':
                shutil.copy2(os.path.join(root, file), 'games.csv')
                print('games.csv copied to current directory!')
                break
else:
    print('games.csv already exists. Skipping download.')

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
# Load the dataset
# Download from: https://www.kaggle.com/datasets/nathanlauga/nba-games
# Place games.csv in the same folder as this notebook
df = pd.read_csv('games.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Check missing values
print('Missing values per column:')
print(df.isnull().sum())
print('\nBasic statistics:')
df.describe()

## 3. Data Preprocessing

In [ ]:
# Select relevant features
features = [
    'FG_PCT_home', 'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home',
    'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away', 'REB_away'
]

# Target: HOME_TEAM_WINS (1 = home wins, 0 = away wins)
target = 'HOME_TEAM_WINS'

# Drop rows with nulls in these columns
df_clean = df[features + [target]].dropna()

print(f'Clean dataset shape: {df_clean.shape}')
print(f'\nTarget distribution:')
print(df_clean[target].value_counts())
print(f'\nHome Win Rate: {df_clean[target].mean()*100:.2f}%')

In [ ]:
# Visualize class distribution
plt.figure(figsize=(6,4))
df_clean[target].value_counts().plot(kind='bar', color=['#e74c3c','#2ecc71'], edgecolor='black')
plt.title('Class Distribution: Home Team Wins')
plt.xlabel('0 = Away Wins, 1 = Home Wins')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('rf_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Split features and target
X = df_clean[features]
y = df_clean[target]

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## 4. Train Random Forest Model

In [ ]:
# Train baseline model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
print('Random Forest model trained!')

In [ ]:
# Hyperparameter tuning with GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV accuracy:', round(grid_search.best_score_ * 100, 2), '%')

# Use best model
best_rf = grid_search.best_estimator_

## 5. Evaluate Model

In [ ]:
# Predictions
y_pred = best_rf.predict(X_test_scaled)
y_prob = best_rf.predict_proba(X_test_scaled)[:, 1]

# Metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 40)
print('   RANDOM FOREST — TEST RESULTS')
print('=' * 40)
print(f'  Accuracy  : {acc*100:.2f}%')
print(f'  Precision : {prec*100:.2f}%')
print(f'  Recall    : {rec*100:.2f}%')
print(f'  F1-Score  : {f1*100:.2f}%')
print(f'  ROC-AUC   : {auc:.4f}')
print('=' * 40)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Away Wins','Home Wins']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Away Wins','Home Wins'],
            yticklabels=['Away Wins','Home Wins'])
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0,1],[0,1],'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest — ROC Curve')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('rf_roc_curve.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance
importances = pd.Series(best_rf.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Random Forest — Feature Importances')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150)
plt.show()

print('Top Features:')
print(importances)

In [ ]:
# Cross-validation
cv_scores = cross_val_score(best_rf, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy (5-fold): {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

## 6. Summary

| Metric | Score |
|--------|-------|
| Accuracy | See above |
| Precision | See above |
| Recall | See above |
| F1-Score | See above |
| ROC-AUC | See above |

**Key Observations:**
- Random Forest is an ensemble method that builds multiple decision trees and aggregates their predictions.
- It handles feature interactions and non-linear relationships well.
- Feature importance analysis reveals which statistics most influence game outcomes.
- Hyperparameter tuning with GridSearchCV improved generalization.
